### Coleta de dados de Temperatura

Será utilizado o dataset derived-era5-single-levels- do ERA5

Documentação em:
https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=documentation

Os dados extraídos:
<pre>
- Temperatura: variável 2m_temperature (t2m), retorna a temperatura em Kelvin
- Para a camada Bronze deverá ser mantida em Kelvin
- Para a camada Silver será necessário conversão (subtrair -273,15)
</pre>

Os dados serão coletados por Ano, Mês e dia

Os dados requisitados estão no retangulo geográfico [6, -74, -34, -31] -> [Norte, Oeste, Sul, Leste] em graus onde está o Brasil

In [1]:
import sys, os
from datetime import datetime
from pyspark.sql import functions as F

In [2]:
# Adiciona a pasta raiz do projeto 
sys.path.append(os.path.abspath(os.path.join('..')))

import copernicus_cds_utils as utils 

In [3]:
# Cria uma conexão Spark 
spark = utils.get_spark_session("Temperatura")

In [4]:
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\"

In [5]:
def transform_data(df_temperatura):
    drop_cols = ["valid_time", "t2m", "number"]

    df_temperatura_final = \
        (df_temperatura
            .withColumns({"data_medicao"    : F.col("valid_time").cast("date")
                         ,"indicador"       : F.lit("temperatura") 
                         ,"valor"           : (F.col("t2m") - F.lit(273.15)).cast("double") # Converte a temperatura de Kelvin para Celsius
                         ,"unidade_medida"  : F.lit("celsius")})
            .drop(*drop_cols)
    )

    df_temperatura_final = \
        (df_temperatura_final
            .select("data_medicao"
                   ,"latitude"
                   ,"longitude"
                   ,"indicador"
                   ,"valor"
                   ,"unidade_medida"))

    return df_temperatura_final

In [6]:
dataset_     = "reanalysis-era5-single-levels"
product_type = "reanalysis"
variable     = "2m_temperature"
ano          = 2026

years_process = range(1995,2027)

for ano in years_process:
    start = datetime(2026, 8, 16).now()
    
    print("Obter dados, converter e gravar para o ano : ",ano, " - ", start, end="" )

    # Recupera os dados do CDS em formato netcdf4 -> Landing
    ret_download = utils.recuperar_dados_ERA5(dataset_, product_type, variable, ano)


    # Somente para processamento LOCAL, deve ser reescrita
    file_name    = r"{DATA_PATH_ROOT}ERA5-temperaturas\arquivos_NC\ERA5_{variable}_{ano}.nc".format(ano = ano, DATA_PATH_ROOT = DATA_PATH_ROOT, variable = variable)
    os.rename(ret_download, file_name)

    # Converte os dados de netcdf4 para um Dataframe Spark  -> Bronze
    df_spark     = utils.converter_netcdf4_Spark_DF(spark, file_name)


    df_temperatura_final = transform_data(df_spark)


    # Escreve os dados em formato csv
    csv_file_name = f"ERA5_t2m_{ano}.csv"
    csv_path = \
        r"{DATA_PATH_ROOT}\ERA5-temperaturas\arquivos_CSV".format(DATA_PATH_ROOT = DATA_PATH_ROOT)


    utils.write_data_csv(df_temperatura_final, csv_path, csv_file_name)



    finish = datetime(2026, 8, 16).now()
    print(f" - Concluído: {csv_file_name} - {finish} - {(finish - start)} \n")




Obter dados, converter e gravar para o ano :  1995  -  2026-08-16 15:02:42.149992

2026-08-16 15:02:44,729 INFO Request ID is ef9b5b53-52b0-4166-9e32-6c6468912460
2026-08-16 15:02:45,530 INFO status has been updated to accepted
2026-08-16 15:03:37,529 INFO status has been updated to running
2026-08-16 15:04:42,002 INFO status has been updated to successful


 - Concluído: ERA5_t2m_1995.csv - 2026-08-16 15:05:40.678180 - 0:02:58.528188 

Obter dados, converter e gravar para o ano :  1996  -  2026-08-16 15:05:40.700480

2026-08-16 15:05:42,644 INFO Request ID is f345f984-1a59-46cd-afc6-193617cd45a4
2026-08-16 15:05:42,842 INFO status has been updated to accepted
2026-08-16 15:06:16,657 INFO status has been updated to running
2026-08-16 15:06:59,789 INFO status has been updated to successful


 - Concluído: ERA5_t2m_1996.csv - 2026-08-16 15:07:47.969857 - 0:02:07.269377 

Obter dados, converter e gravar para o ano :  1997  -  2026-08-16 15:07:47.985483

...

 - Concluído: ERA5_t2m_2024.csv - 2026-08-16 16:40:13.115390 - 0:03:03.251663 

Obter dados, converter e gravar para o ano :  2025  -  2026-08-16 16:40:13.122393

2026-08-16 16:40:15,055 INFO Request ID is 2bad9b7c-60e3-4624-bcee-c5a0cbc26bd6
2026-08-16 16:40:15,392 INFO status has been updated to accepted
2026-08-16 16:40:50,168 INFO status has been updated to running
2026-08-16 16:42:11,920 INFO status has been updated to successful


 - Concluído: ERA5_t2m_2025.csv - 2026-08-16 16:43:16.655729 - 0:03:03.533336 

Obter dados, converter e gravar para o ano :  2026  -  2026-08-16 16:43:16.660014

2026-08-16 16:43:18,527 INFO Request ID is 7dfcb6c4-2fb5-4f66-a03a-0a6afb5558c7
2026-08-16 16:43:18,727 INFO status has been updated to accepted
2026-08-16 16:44:09,520 INFO status has been updated to running
2026-08-16 16:44:35,355 INFO status has been updated to successful


 - Concluído: ERA5_t2m_2026.csv - 2026-08-16 16:44:55.920621 - 0:01:39.260607 



In [7]:
# df_temperatura_final.printSchema()
# df_temperatura_final.show(10, False)